# KubeCon India 2026 — Platform + App Setup (GCP / KCC track)

The notebook form of [`setup-with-kcc.sh`](setup-with-kcc.sh) — run **after**
[`00_init-with-kcc.ipynb`](00_init-with-kcc.ipynb). Applies the KCC backing of the
`bucket` claim and deploys the GCP Application. Like ACK (and unlike Crossplane), KCC
has no composition layer, so Step 1 is one `vela def apply`.

## Steps
1. Apply the `bucket` backing (`bucket-kcc.cue`)
2. Build + push the app images (product-catalog + bucket-browser)
3. Create GCP credentials in the app namespaces (dev / staging / prod)
4. Deploy the GCP Application (`product-catalog-gcp.yaml`)
5. Verify


## Step 1: Apply the `bucket` backing (the How)

A single `vela def apply` of the KCC backing — it resolves the same `bucket` claim into
a GCS `StorageBucket` (mapping the claim's AWS-style `region` to a GCP `location`).

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"

print_step "Applying the 'bucket' ComponentDefinition (KCC backing)"
vela def apply "$REPO_ROOT/platform/kubevela/components/bucket-kcc.cue"
print_success "'bucket' ComponentDefinition (KCC) installed"


## Step 2: Build + push the app images

Build both images — the **product-catalog** API and the **bucket-browser** web UI (the
two `webservice` components). The apps are cloud-neutral; the same images run on AWS or
GCP.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
bash "$REPO_ROOT/apps/product-catalog/build-image.sh"
bash "$REPO_ROOT/apps/bucket-browser/build-image.sh"


## Step 3: GCP credentials in the app namespaces

App pods mount a `gcp-key` secret (data key `key.json`) to reach GCS — the app's runtime
creds, separate from the KCC controller's secret in `cnrm-system`. Load `.env.gcp`, then
create the secret per env.

In [ ]:
%%bash
set -euo pipefail
REPO_ROOT="$(cd ../../.. && pwd)"
source "$REPO_ROOT/scripts/common.sh"
source "$REPO_ROOT/scripts/load-gcp-env.sh"
source "$REPO_ROOT/scripts/create-gcp-secret.sh"

load_gcp_env "$PWD/.env.gcp"
for ns in dev staging prod; do
    create_gcp_secret --create-namespace "$ns" gcp-key
done


## Step 4: Deploy the GCP Application

Deploy `product-catalog-gcp.yaml` — structurally identical to the AWS Application; only
the cloud-runtime block differs (STORAGE_PROVIDER=gcp, gcp-key mount). The `bucket` claim
is byte-for-byte identical and now resolves via KCC into a GCS bucket.

In [ ]:
%%bash
set -euo pipefail
DEMO_DIR="$(cd .. && pwd)"
source "$(cd ../../.. && pwd)/scripts/common.sh"

vela up -f "$DEMO_DIR/kubevela/product-catalog-gcp.yaml"
print_success "Application submitted (workflow deploys dev, then suspends for approval)"


## Step 5: Verify

The workflow's `request` steps already exercised the API against its GCS bucket. The
bucket-browser UI lets you see the objects — same image/UI as the AWS track.

In [ ]:
%%bash
set -euo pipefail
vela status product-catalog || true
echo
kubectl get pods,hpa -n dev
echo
echo "The KCC-provisioned GCS bucket(s):"
kubectl get storagebucket -A || true
echo
echo "Browse the objects (run in a separate terminal):"
echo "  kubectl -n dev port-forward svc/bucket-browser 8080:8080   # http://localhost:8080"


## Setup complete

The KCC track is deployed (API + bucket-browser + a GCS bucket per env).

### Teardown
KCC teardown is clean — the `StorageBucket` carries the `force-destroy` annotation, so
deleting the Application empties and removes the bucket. Then delete the cluster:
```bash
vela delete product-catalog -y
../cleanup.sh
```
